# Manejo de Memoria y Punteros en Rust

Rust ofrece un control sobre la memoria similar a C++, pero con garantías de seguridad que evitan errores comunes como punteros nulos o colgados. Para entender Rust, es vital comprender cómo se gestiona la memoria en el **Stack** y el **Heap**, y cómo los diferentes tipos de punteros afectan la definición de variables.

## 1. El Stack vs El Heap

### El Stack (Pila)
- **Rápido**: El acceso es inmediato (LIFO).
- **Tamaño Fijo**: Solo datos cuyo tamaño se conoce en tiempo de compilación.
- **Gestión Automática**: Los datos se eliminan cuando la función sale de ámbito.

### El Heap (Montículo)
- **Flexible**: Para datos cuyo tamaño puede cambiar (ej. `String`, `Vec`).
- **Más Lento**: Requiere buscar un lugar libre y seguir un puntero.
- **Gestión por Ownership**: Rust libera la memoria del heap automáticamente cuando el dueño sale de ámbito.

In [10]:
fn main() {
    let x = 5;            // Stack: entero de tamaño fijo
    let y = x;            // Stack: se copia el valor (Copy trait)

    let s1 = String::from("hola"); // s1 (puntero) en Stack, "hola" en Heap
    let s2 = s1;          // MOVE: s2 recibe el puntero, s1 ya no es válida
    println!("valor: {}", y);
}

main();

valor: 5


## 2. Tipos de Punteros en Rust

### A. Referencias (`&T` y `&mut T`)
Son "punteros seguros". El compilador garantiza que siempre apunten a memoria válida mediante el *Borrow Checker*.

In [ ]:
let mut valor = 10;
let r1 = &valor;     // Referencia inmutable
let r2 = &mut valor; // Referencia mutable (solo una permitida a la vez)

// r1 ya no se puede usar porque r2 tomó un préstamo mutable

### B. Smart Pointers (Punteros Inteligentes)
Son estructuras que actúan como punteros pero tienen metadatos y capacidades extra.

#### 1. `Box<T>`
Se usa para poner datos en el heap. Es el puntero inteligente más simple. Define una propiedad única.

In [ ]:
let b = Box::new(5); // El 5 vive en el Heap
println!("b = {}", b);

// Caso de uso: Estructuras recursivas (tamaño desconocido)
enum List {
    Cons(i32, Box<List>),
    Nil,
}

#### 2. `Rc<T>` y `Arc<T>` (Reference Counting)
Permiten **propiedad múltiple**. El dato en el heap no se libera hasta que el último puntero desaparece.
- `Rc`: Para un solo hilo (rápido).
- `Arc`: Atómico, para múltiples hilos.

In [ ]:
use std::rc::Rc;

let data = Rc::new(String::from("compartido"));
let clon1 = Rc::clone(&data);
let clon2 = Rc::clone(&data);
println!("Referencias: {}", Rc::strong_count(&data));

#### 3. `RefCell<T>` (Mutabilidad Interior)
Permite modificar datos incluso si tenemos una referencia inmutable al contenedor. Las reglas de préstamo se chequean en **tiempo de ejecución** (si fallan, hay un panic).

In [ ]:
use std::cell::RefCell;

let x = RefCell::new(5);
{
    let mut m = x.borrow_mut();
    *m += 1;
}
println!("Valor: {:?}", x.borrow());

## 3. Raw Pointers (Punteros Crudos: `*const T` y `*mut T`)

Se usan en código `unsafe`. Pueden ser nulos, no tienen garantías de validez y pueden causar fugas de memoria si no se manejan con cuidado. Se usan principalmente para FFI (interfaz con C) o estructuras de datos de muy bajo nivel.

In [ ]:
let mut num = 5;

let r1 = &num as *const i32;
let r2 = &mut num as *mut i32;

unsafe {
    println!("r1: {}", *r1);
    *r2 = 10;
    println!("num ahora es: {}", num);
}

## 4. Influencia en la Definición de Tipos

En Rust, el tipo de puntero que elijas cambia drásticamente la semántica de la variable:

1.  **`T`**: Propiedad única. Si asignas, mueves (`Move`).
2.  **`&T`**: Préstamo inmutable. Solo lectura.
3.  **`&mut T`**: Préstamo mutable exclusivo.
4.  **`Box<T>`**: Propiedad única en el Heap.
5.  **`Rc<T>`**: Propiedad compartida (solo lectura).
6.  **`Rc<RefCell<T>>`**: El patrón estándar para propiedad compartida y mutable al mismo tiempo (ej. en árboles o grafos).

### El impacto de Sized
Rust necesita saber el tamaño de los tipos. Tipos como `[i32]` (slice) o `dyn Trait` son **DST (Dynamically Sized Types)**. No pueden vivir solos en el stack; siempre deben estar detrás de un puntero (`&[i32]`, `Box<dyn Trait>`).

## 5. Resumen de Elección de Punteros

| Necesidad | Puntero Recomendado |
| :--- | :--- |
| Pasar dato sin copiar (lectura) | `&T` |
| Modificar dato ajeno | `&mut T` |
| Guardar en heap por tamaño | `Box<T>` |
| Múltiples dueños (1 hilo) | `Rc<T>` |
| Múltiples dueños (Multi-hilo) | `Arc<T>` |
| Mutabilidad interior | `RefCell<T>` (o `Mutex<T>` en hilos) |